# PGA-UNet — Train (smoke test cho repo `PGA_Unet2D`)

Notebook này clone trực tiếp từ `https://github.com/ThongLuc2k3/PGA_Unet2D.git` (repo đang chuẩn bị push) và chạy package `Source/Prompt-Guided-XRay-Segmentation/` y như trên máy local, để kiểm tra code có hoạt động đúng sau khi đẩy lên GitHub hay không.

**Chạy tuần tự:** Setup → (tuỳ chọn) Quick smoke run 2 epoch → Train đầy đủ.

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 — SETUP: clone repo, tải dataset, cài dependency
# ══════════════════════════════════════════════════════
import os, torch

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO      = 'https://github.com/ThongLuc2k3/PGA_Unet2D.git'
REPO_DIR  = f'{BASE}/PGA_Unet2D'
PGA_ROOT  = f'{REPO_DIR}/Source/Prompt-Guided-XRay-Segmentation'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -q {REPO} {REPO_DIR}')
print('  \u2705 repo cloned:', REPO_DIR)

DS_ZIP = f'{BASE}/dataset_BTXRD.zip'
if not os.path.exists(DS_ZIP):
    os.system('pip install -q gdown')
    import gdown
    gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                   DS_ZIP, quiet=False)
if not os.path.exists(f'{PGA_ROOT}/dataset_BTXRD'):
    os.system(f'unzip -oq {DS_ZIP} -d {PGA_ROOT}/')
print('  \u2705 dataset ready at', f'{PGA_ROOT}/dataset_BTXRD')

os.system('pip install -q tqdm opencv-python matplotlib scipy')

os.chdir(PGA_ROOT)
print(f'\n\u2705 Setup DONE  |  cwd={os.getcwd()}  |  device={"cuda" if torch.cuda.is_available() else "cpu"}')

## Quick smoke run (tuỳ chọn, khuyên chạy trước)

Chạy training thật trên dữ liệu thật nhưng chỉ 2 epoch / batch nhỏ, để xác nhận toàn bộ pipeline (dataset → model → loss → backward → checkpoint) chạy được trên Colab **trước khi** chạy full 100 epoch. Không đụng tới `train.py` gốc — chỉ import lại các thành phần và tự viết loop rút gọn.

In [ ]:
# ══════════════════════════════════════════════════════
# QUICK SMOKE RUN — 2 epoch, batch nhỏ, chỉ để verify pipeline
# ══════════════════════════════════════════════════════
import sys, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader

if PGA_ROOT not in sys.path:
    sys.path.insert(0, PGA_ROOT)

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

smoke_train_ds = BTXRD_Dataset(
    image_dir=f'{PGA_ROOT}/dataset_BTXRD/train/images',
    json_dir=f'{PGA_ROOT}/dataset_BTXRD/train/annotations',
    img_size=256, is_train=True, prompt_mode='mixed_7_3')
smoke_loader = DataLoader(smoke_train_ds, batch_size=4, shuffle=True, num_workers=2)

smoke_model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
criterion   = nn.BCEWithLogitsLoss()
optimizer   = optim.AdamW(smoke_model.parameters(), lr=1e-4, weight_decay=1e-4)

def dice_loss(pred, target, smooth=1e-5):
    pred_soft = torch.sigmoid(pred)
    inter = (pred_soft * target).sum(dim=(1, 2, 3))
    union = pred_soft.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return (1 - ((2. * inter + smooth) / (union + smooth))).mean()

smoke_model.train()
for epoch in range(2):
    for it, (images, masks, prompts) in enumerate(smoke_loader):
        images, masks, prompts = images.to(DEVICE), masks.to(DEVICE), prompts.to(DEVICE)
        preds = smoke_model(images, prompts)
        loss  = criterion(preds, masks) + dice_loss(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), max_norm=1.0)
        optimizer.step()
        if it >= 3:
            break
    print(f'epoch {epoch}  loss={loss.item():.4f}')

print('\n\u2705 Quick smoke run OK \u2014 pipeline hoạt động, có thể chạy train.py đầy đủ.')

## Train đầy đủ

Trước khi chạy, mở `train.py` và chỉnh 2 dòng cấu hình đầu file nếu cần:
- `EXPERIMENT = 'A'` (zoom_out only) hoặc `'B'` (mixed 70/30, mặc định)
- `USE_ENCODER_PROMPT = True/False`

Checkpoint sẽ lưu ở `checkpoints/pga_unet_exp{EXPERIMENT}_best.pth`.

In [ ]:
# ══════════════════════════════════════════════════════
# TRAIN ĐẦY ĐỦ (dùng đúng script train.py trong repo)
# ══════════════════════════════════════════════════════
os.chdir(PGA_ROOT)
!python train.py